In [1]:

import pandas as pd
import numpy as np
import geopandas as gpd
import glob
from shapely import wkt

# 분석기간 내 부산 전체 성연령, 시간대 카드이용 데이터 리스트
data_path = glob.glob(r"D:\\02_사업관리\\자체분석\\02_카드이용\\*.csv", recursive=True)
output_path = r"D:\\02_사업관리\\2026년\\데이터 분석 컨설팅\\20260619_클래식부산\\02_집계결과\\"
# 100m격자, 분석영역, 연령코드 데이터 불러오기
grid100 = gpd.read_file(r"D:\\00_자료검토\\03_공간자료\\griddata\\grid_마라_100M.shp", crs=5179)
area = pd.read_csv(r"D:\\02_사업관리\\2026년\\데이터 분석 컨설팅\\20260619_클래식부산\\output_area_20260619_v1.csv", sep="|", encoding="utf-8")
agecode = pd.read_csv(r"D:\\02_사업관리\\코드자료\\bccd_age_code.csv", sep="|", dtype={'age_cd': str}, encoding="utf-8")
agecode.rename(columns={'age_cd': 'ages'}, inplace=True)

# 분석기간 내 분석지역(부산진구) 성연령, 시간대 카드이용 데이터 추출
emd_lists = []
tme_lists = []
for data in data_path:
    if '_emd_' in data:
        df = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str, 'ages': str}, encoding="utf-8")
        emd_lists.append(df[df['hcode'].str.startswith('26230')].copy())
        emd= pd.concat(emd_lists, ignore_index=True)
    else:
        df = pd.read_csv(data, sep="|", dtype={'std_ymd': str, 'hcode': str, 'tme': str}, encoding="utf-8")
        tme_lists.append(df[df['hcode'].str.startswith('26230')].copy())
        tme = pd.concat(tme_lists, ignore_index=True)

# 분석대상영역기준으로 추출
# 분석대상영역을 GeoDataFrame으로 변환
def geopreprocess(df):
    return gpd.GeoDataFrame(df, geometry=wkt.loads(df['geometry']), crs=5179)
area_gdf = geopreprocess(area)
# print(area_gdf)
# 분석대상영역 내 100m 격자ID 추출
grid100['x'] = grid100.geometry.centroid.x.round(3)
grid100['y'] = grid100.geometry.centroid.y.round(3)
grid100_pt = gpd.GeoDataFrame(grid100, geometry=gpd.points_from_xy(grid100.x, grid100.y), crs=5179)
area_g100_pt = gpd.sjoin(grid100_pt, area_gdf, how="inner", predicate="intersects")
area_g100_pt.rename(columns={'GRID_100M_': 'id'}, inplace=True)
area_g100_pt = area_g100_pt[['id', 'areaname', 'geometry']]
# print(area_g100_pt)

# 분석용 컬럼 전처리
def preprocess(df):
    df = pd.merge(area_g100_pt, df, on="id", how="left").dropna()
    df['year'] = df['std_ymd'].astype('datetime64[ns]').astype(str).str[:4] + "년"
    df['month'] = df['std_ymd'].astype('datetime64[ns]').astype(str).str[5:7] + "월"
    if 'gender' in df.columns:
        df = pd.merge(df, agecode, on="ages", how="left")
        df.loc[df['gender'].str.contains('M|m'), '성별'] = "남성"
        df.loc[df['gender'].str.contains('F|f'), '성별'] = "여성"
        df.loc[df['age_nm'].str.contains('20세미만'), "연령"] = '20대미만'
        df.loc[df['age_nm'].str.contains('20~24|25~29'), "연령"] = '20대'
        df.loc[df['age_nm'].str.contains('30~34|35~39'), "연령"] = '30대'
        df.loc[df['age_nm'].str.contains('40~44|45~49'), "연령"] = '40대'
        df.loc[df['age_nm'].str.contains('50~54|55~59'), "연령"] = '50대'
        df.loc[df['age_nm'].str.contains('60~64|65~69'), "연령"] = '60대'
        df.loc[df['age_nm'].str.contains('70~74|75~79'), "연령"] = '70대'
        df.loc[df['age_nm'].str.contains('80~84|85세이상'), "연령"] = '80대이상'
    if 'tme' in df.columns:
        df['시간'] = df['tme'] + "시"
        df.loc[df['tme'].str.contains('00|01|02|03'), '시간대'] = '심야'
        df.loc[df['tme'].str.contains('04|05|06|07'), '시간대'] = '새벽'
        df.loc[df['tme'].str.contains('08|09|10|11'), '시간대'] = '오전'
        df.loc[df['tme'].str.contains('12|13|14|15'), '시간대'] = '오후'
        df.loc[df['tme'].str.contains('16|17|18|19'), '시간대'] = '저녁'
        df.loc[df['tme'].str.contains('20|21|22|23'), '시간대'] = '야간'
    return df

genders = preprocess(emd).groupby(
    ['year','month','id','성별'], as_index=False).agg(
        {'amt': np.sum, 'cnt': np.sum}).sort_values(['year','month'], ascending=True)
genders.columns = ['기준연','기준월','id','성별','매출금액','매출건수']
genders.to_excel(output_path + "월별 성별 카드이용 현황_분석기간전체.xlsx", engine="openpyxl", index=False)
ages = preprocess(emd).groupby(
    ['year','month','id','연령'], as_index=False).agg(
        {'amt': np.sum, 'cnt': np.sum}).sort_values(['year','month'], ascending=True)
ages.columns = ['기준연','기준월','id','연령','매출금액','매출건수']
ages.to_excel(output_path + "월별 연령 카드이용 현황_분석기간전체.xlsx", engine="openpyxl", index=False)
times = preprocess(tme).groupby(
    ['year','month','id','시간','시간대'], as_index=False).agg(
        {'amt': np.sum, 'cnt': np.sum}).sort_values(['year','month'], ascending=True)
times.columns = ['기준연','기준월','id','시간','시간대','매출금액','매출건수']
times.to_excel(output_path + "월별 시간 카드이용 현황_분석기간전체.xlsx", engine="openpyxl", index=False)


c:\miniconda3\envs\machine\lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver ESRI Shapefile does not support open option CRS
  return ogr_read(
C:\Users\COMTREE\AppData\Local\Temp\ipykernel_12040\238987606.py:72: FutureWarning: The provided callable <function sum at 0x0000026B48A75F30> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  ['year','month','id','성별'], as_index=False).agg(
C:\Users\COMTREE\AppData\Local\Temp\ipykernel_12040\238987606.py:72: FutureWarning: The provided callable <function sum at 0x0000026B48A75F30> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  ['year','month','id','성별'], as_index=False).agg(
C:\Users\COMTREE\AppData\Local\Temp\ipykernel_12040\238987606.py:77: FutureWarning: The provided callable <function s

In [ ]:
# 시각화
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

pd.options.display.float_format = '{:.0f}'.format

plt.rc('font', size=10)
gpkg_path = r"D:\\02_사업관리\\2026년\\데이터 분석 컨설팅\\20260619_클래식부산\\2026_017_부산콘서트홀_20260619.gpkg"

area_g100 = gpd.read_file(gpkg_path, layer="area_g100")
males = genders[genders['성별'] == '남성'].copy()
females = genders[genders['성별'] == '여성'].copy()
year_males_amt = (males.groupby(['기준연','id'])['매출금액'].sum() / 1000000).round()
year_males_amt = year_males_amt.unstack(level=0).fillna(0)
year_males_amt = pd.merge(area_g100, year_males_amt, on="id", how="inner")
year_males_amt.to_file(gpkg_path, layer='yearly_males_amt', driver='GPKG')

year_males_cnt = (males.groupby(['기준연','id'])['매출건수'].sum() / 10000).round()
year_males_cnt = year_males_amt.unstack(level=0).fillna(0)
year_males_cnt = pd.merge(area_g100, year_males_amt, on="id", how="inner")
year_males_cnt.to_file(gpkg_path, layer='yearly_males_cnt', driver='GPKG')

c:\miniconda3\envs\machine\lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: This version of GeoPackage user_version=0x000028A0 (10400, v1.4.0) on 'D:\\02_사업관리\\2026년\\데이터 분석 컨설팅\\20260619_클래식부산\\2026_017_부산콘서트홀_20260619.gpkg' may only be partially supported
  return ogr_read(
c:\miniconda3\envs\machine\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: This version of GeoPackage user_version=0x000028A0 (10400, v1.4.0) on 'D:\\02_사업관리\\2026년\\데이터 분석 컨설팅\\20260619_클래식부산\\2026_017_부산콘서트홀_20260619.gpkg' may only be partially supported
  ogr_write(
